In [86]:
# Library yang digunakan
from nltk.tokenize import word_tokenize
import string
import pandas as pd
import numpy as np
import nltk
import emoji
import re
from sklearn.pipeline import Pipeline
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mirur\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## 💡 Load Data

In [87]:
#Load Dataser
version = '5.3'
file_path = f"../data/review_genshin_{version}_cleaned.csv"
data = pd.read_csv(file_path)

In [88]:
# Show Dataset
data.head(10)

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
0,869d8e47-1649-4823-a630-dd9dea341421,Good,5,0,2025-02-10 22:52:05,5.3
1,a885d7e9-e655-4583-9686-488fe34938ad,Moga menang rate off,5,0,2025-02-10 22:21:59,5.3
2,59910aae-3174-4ab6-8b9f-eb7d6ded0ffb,Udah ganti aja Jagan jadi game petualangan jad...,1,1,2025-02-10 22:05:06,5.3
3,5800da6b-abcb-48f6-8a63-6564c452d03b,"Menurutku sih bagus² aja game ini, game petual...",5,0,2025-02-10 20:54:30,5.3
4,63ca4e9b-0282-4e85-8f66-618a28c42628,Terlalu besar,4,0,2025-02-10 19:13:14,5.3
5,8cc8b53a-1be8-4071-8bf9-c9ca47bdfe7f,good,5,0,2025-02-10 18:47:56,5.3
6,c2c126c1-b22c-4cf5-854a-d3148abcd9c4,Saya benci game ini karna kalah rate off di ba...,1,2,2025-02-10 18:14:05,5.3
7,f37248a9-4043-4711-b143-e8ecb44204ce,jelek,1,1,2025-02-10 17:50:14,5.3
8,679cadaa-6195-4ed7-a4ec-3ecf2325a388,Game kikir,2,0,2025-02-10 17:40:43,5.3
9,5cde3567-3ff9-4660-8e29-dcf424b36dae,game kikir capek gw kalah rate off terus terus...,1,1,2025-02-10 17:36:10,5.3


In [89]:
data = data[data['Score'].between(1, 3)]

In [90]:
# proses case folding 
def casefolding(Content):
    Content = Content.lower()
    return Content
data['Content'] = data['Content'].apply(casefolding)
data.head()

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
2,59910aae-3174-4ab6-8b9f-eb7d6ded0ffb,udah ganti aja jagan jadi game petualangan jad...,1,1,2025-02-10 22:05:06,5.3
6,c2c126c1-b22c-4cf5-854a-d3148abcd9c4,saya benci game ini karna kalah rate off di ba...,1,2,2025-02-10 18:14:05,5.3
7,f37248a9-4043-4711-b143-e8ecb44204ce,jelek,1,1,2025-02-10 17:50:14,5.3
8,679cadaa-6195-4ed7-a4ec-3ecf2325a388,game kikir,2,0,2025-02-10 17:40:43,5.3
9,5cde3567-3ff9-4660-8e29-dcf424b36dae,game kikir capek gw kalah rate off terus terus...,1,1,2025-02-10 17:36:10,5.3


In [91]:
def cleansing(Content):
    Content = emoji.replace_emoji(Content, replace='')  # Hapus semua emoji
    Content = Content.strip(" ")
    Content = re.sub(r'[?|$|.|!_:")(-+,]', '', Content)
    Content = re.sub(r'\d+', '', Content)
    Content = re.sub(r"\b[a-zA-Z]\b", "", Content)
    Content = re.sub(r'\s+', ' ', Content)
    return Content

data['Content'] = data['Content'].apply(cleansing)

In [92]:
# NLTK word tokenize


def word_tokenize_wrapper(text):
    return word_tokenize(text)


data['Content'] = data['Content'].apply(word_tokenize_wrapper)
data.head()

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
2,59910aae-3174-4ab6-8b9f-eb7d6ded0ffb,"[udah, ganti, aja, jagan, jadi, game, petualan...",1,1,2025-02-10 22:05:06,5.3
6,c2c126c1-b22c-4cf5-854a-d3148abcd9c4,"[saya, benci, game, ini, karna, kalah, rate, o...",1,2,2025-02-10 18:14:05,5.3
7,f37248a9-4043-4711-b143-e8ecb44204ce,[jelek],1,1,2025-02-10 17:50:14,5.3
8,679cadaa-6195-4ed7-a4ec-3ecf2325a388,"[game, kikir]",2,0,2025-02-10 17:40:43,5.3
9,5cde3567-3ff9-4660-8e29-dcf424b36dae,"[game, kikir, capek, gw, kalah, rate, off, ter...",1,1,2025-02-10 17:36:10,5.3


In [93]:
normalizad_word = pd.read_csv("../data/normalisasi.csv")

normalizad_word_dict = {}

for index, row in normalizad_word.iterrows():
    if row[0] not in normalizad_word_dict:
        normalizad_word_dict[row[0]] = row[1] 

def normalized_term(document):
    return [normalizad_word_dict[term] if term in normalizad_word_dict else term for term in document]

data['Content'] = data['Content'].apply(normalized_term)

data.head()

C:\Users\mirur\AppData\Local\Temp\ipykernel_1792\2302089821.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[0] not in normalizad_word_dict:
C:\Users\mirur\AppData\Local\Temp\ipykernel_1792\2302089821.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  normalizad_word_dict[row[0]] = row[1]


,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
2,59910aae-3174-4ab6-8b9f-eb7d6ded0ffb,"[sudah, ganti, saja, jagan, jadi, game, petual...",1,1,2025-02-10 22:05:06,5.3
6,c2c126c1-b22c-4cf5-854a-d3148abcd9c4,"[saya, benci, game, ini, karena, kalah, rate, ...",1,2,2025-02-10 18:14:05,5.3
7,f37248a9-4043-4711-b143-e8ecb44204ce,[jelek],1,1,2025-02-10 17:50:14,5.3
8,679cadaa-6195-4ed7-a4ec-3ecf2325a388,"[game, kikir]",2,0,2025-02-10 17:40:43,5.3
9,5cde3567-3ff9-4660-8e29-dcf424b36dae,"[game, kikir, capek, saya, kalah, rate, off, t...",1,1,2025-02-10 17:36:10,5.3


In [94]:

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
sw = pd.read_csv("../data/stopwords_id.csv")


def stopword_removal(Content):
    filtering = stopwords.words('indonesian', 'english')
    filtering.extend(sw)
    x = []
    data = []

    def myFunc(x):
        if x in filtering:
            return False
        else:
            return True
    fit = filter(myFunc, Content)
    for x in fit:
        data.append(x)
    return data


data['Content'] = data['Content'].apply(stopword_removal)
data.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mirur\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
2,59910aae-3174-4ab6-8b9f-eb7d6ded0ffb,"[ganti, jagan, game, petualangan, game, cerita...",1,1,2025-02-10 22:05:06,5.3
6,c2c126c1-b22c-4cf5-854a-d3148abcd9c4,"[benci, game, kalah, rate, off, banner, weapon...",1,2,2025-02-10 18:14:05,5.3
7,f37248a9-4043-4711-b143-e8ecb44204ce,[jelek],1,1,2025-02-10 17:50:14,5.3
8,679cadaa-6195-4ed7-a4ec-3ecf2325a388,"[game, kikir]",2,0,2025-02-10 17:40:43,5.3
9,5cde3567-3ff9-4660-8e29-dcf424b36dae,"[game, kikir, capek, kalah, rate, off, terusan...",1,1,2025-02-10 17:36:10,5.3


### Proses Stemming dan Membuat file data baru (dataset yang sudah dibersihkan melalui proses NLTK)

In [95]:
# proses stemming
def stemming(Content):
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()
    do = []
    for w in Content:
        dt = stemmer.stem(w)
        do.append(dt)
    d_clean = []
    d_clean = " ".join(do)
    print(d_clean)
    return d_clean


data['Content'] = data['Content'].apply(stemming)

data.to_csv(f'../data/review_genshin_{version}_clear.csv', index=False)
data_clean = pd.read_csv(
    f'../data/review_genshin_{version}_clear.csv', encoding='latin1')
data_clean.head()

ganti jagan game tualang game cerita banyak bacot skip
benci game kalah rate off banner weapon karakter pacth
jelek
game kikir
game kikir capek kalah rate off terus kalah rate off hard pity
tombol skip story makan main cepat
kurang woy dialong sampah
server benerin bang jaring kenceng rossi main ghensin ngedrop
game
qol bad
tarik mb nya
dar saran fp tinggal game main biar nyesel download size gede gaban
cari material susah habis muncul badmood
please hoyoverse make it easy to find artifacts with substats and stats that always contain critical rate and critical damagelest the genshin game be considered the most miserly game ever because it always bothers its players in every way
butuh fitur skip cakap
voice les karakter dub english
bagus sih pelit banget sistem gacha dasar kikir
jelek
banyak yapping
tolong cut scene dialog gua main game nontonin cutscene abis nonton mending gua nonton film
have so much expectation for the natlan archon quest but comparing to what we got at fontaine and 

,ReviewID,Content,Score,ThumbsUpCount,at,AppVersion
0,59910aae-3174-4ab6-8b9f-eb7d6ded0ffb,ganti jagan game tualang game cerita banyak ba...,1,1,2025-02-10 22:05:06,5.3
1,c2c126c1-b22c-4cf5-854a-d3148abcd9c4,benci game kalah rate off banner weapon karakt...,1,2,2025-02-10 18:14:05,5.3
2,f37248a9-4043-4711-b143-e8ecb44204ce,jelek,1,1,2025-02-10 17:50:14,5.3
3,679cadaa-6195-4ed7-a4ec-3ecf2325a388,game kikir,2,0,2025-02-10 17:40:43,5.3
4,5cde3567-3ff9-4660-8e29-dcf424b36dae,game kikir capek kalah rate off terus kalah ra...,1,1,2025-02-10 17:36:10,5.3
